# 01b — Consistency Analysis

Test LLM prediction stability by running the **same model** on the **same 100 loans** multiple times.

**Goal:** Measure how many predictions flip across runs due to LLM non-determinism.

## Setup

Imports and config. One model (GPT-5.4), two conditions, `N_RUNS=3` each — 6 runs total. Each run gets a distinct label so its cache and `llm_calls.csv` rows never collide.

In [1]:
# llm_utils.py and llm_pricing.py live one directory up — make them importable.
import sys; sys.path.insert(0, "..")
import os
import pandas as pd
import numpy as np

from llm_utils import (
    load_llm_sample, run_ml_on_sample, run_llm_experiment,
    evaluate_predictions, RESULTS_DIR
)

In [2]:
FORCE_RERUN = False  # Set to True to re-run API calls from scratch
# ── Configuration ──────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

API_PROVIDER = "openai"
MODEL_NAME   = "gpt-5.4"
LABEL        = "GPT-5.4"
CONDITIONS   = [False, True]       # no_desc and with_desc
N_RUNS       = 3                   # per condition

from llm_utils import load_all_api_keys
API_KEYS = load_all_api_keys(API_PROVIDER)
MAX_WORKERS = max(1, len(API_KEYS) * 4)
print(f"Using {len(API_KEYS)} API key(s) with {MAX_WORKERS} workers per run.")

Using 3 API key(s) across 6 parallel runs (2 runs per key).


## Load Data

The same 100-loan `tuning_sample` feeds every run — that's the point: hold the input fixed and vary only the model's own randomness. XGBoost is scored once, for reference.

In [3]:
llm_sample = load_llm_sample()
y_true = llm_sample['loan_status'].values

xgb_probs, xgb_preds = run_ml_on_sample(llm_sample)

print(f"Sample size: {len(llm_sample)}")
print(f"Running {N_RUNS} runs × {len(CONDITIONS)} conditions = {N_RUNS * len(CONDITIONS)} total runs of {LABEL}")

Sample size: 100
Running 3 runs × 2 conditions = 6 total runs of GPT-5.4


## Run Multiple Times

All 6 runs (3 × 2 conditions) launch in parallel. Since the prompt and data are identical across a condition's three runs, any prediction that differs between them is pure LLM non-determinism.

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_one(run_idx, include_desc):
    desc_tag = "with_desc" if include_desc else "no_desc"
    return run_llm_experiment(
        use_cache=not FORCE_RERUN,
        llm_sample,
        api_provider=API_PROVIDER,
        model_name=MODEL_NAME,
        api_keys=API_KEYS,
        max_workers=MAX_WORKERS,   # distribute across 2 keys
        include_desc=include_desc,
        label=f"{LABEL} Run {run_idx + 1} ({desc_tag})",
    )

# Launch all 6 runs in parallel
jobs = [(r, d) for d in CONDITIONS for r in range(N_RUNS)]
all_runs = {}

with ThreadPoolExecutor(max_workers=len(jobs)) as pool:
    futures = {
        pool.submit(run_one, r, d): (r, d) for r, d in jobs
    }
    for future in as_completed(futures):
        run_idx, include_desc = futures[future]
        desc_tag = "with_desc" if include_desc else "no_desc"
        result = future.result()
        all_runs[(desc_tag, run_idx)] = result

# Organize into lists per condition
runs_no_desc = [all_runs[('no_desc', i)] for i in range(N_RUNS)]
runs_with_desc = [all_runs[('with_desc', i)] for i in range(N_RUNS)]

print(f"\nAll {len(all_runs)} runs complete.")

[GPT-5.4 Run 2 (no_desc) | no_desc] First call OK (pred=1, prob=1.0). Starting full run...
[GPT-5.4 Run 3 (no_desc) | no_desc] First call OK (pred=1, prob=1.0). Starting full run...
[GPT-5.4 Run 1 (no_desc) | no_desc] First call OK (pred=1, prob=1.0). Starting full run...
[GPT-5.4 Run 2 (with_desc) | with_desc] First call OK (pred=1, prob=1.0). Starting full run...
[GPT-5.4 Run 1 (with_desc) | with_desc] First call OK (pred=1, prob=1.0). Starting full run...
[GPT-5.4 Run 3 (with_desc) | with_desc] First call OK (pred=1, prob=1.0). Starting full run...
[GPT-5.4 Run 2 (with_desc) | with_desc] 10/100 done (20s elapsed, ~183s remaining)
[GPT-5.4 Run 3 (no_desc) | no_desc] 10/100 done (21s elapsed, ~190s remaining)
[GPT-5.4 Run 1 (with_desc) | with_desc] 10/100 done (21s elapsed, ~187s remaining)
[GPT-5.4 Run 3 (with_desc) | with_desc] 10/100 done (21s elapsed, ~186s remaining)
[GPT-5.4 Run 1 (no_desc) | no_desc] 10/100 done (22s elapsed, ~194s remaining)
[GPT-5.4 Run 2 (no_desc) | no_desc]

## Stability Analysis

Per condition: how many of the 100 loans get the **same** label across all three runs (stable) vs flip at least once (unstable), the majority-vote accuracy, and the per-run metric spread. A small spread means a single run is a fair report of the model.

In [5]:
# Stability analysis per condition
for cond_name, runs in [('no_desc', runs_no_desc), ('with_desc', runs_with_desc)]:
    print(f"\n{'='*60}")
    print(f"Stability: {LABEL} ({cond_name})")
    print(f"{'='*60}")

    pred_matrix = pd.DataFrame({
        f'run_{i+1}': r['predictions'] for i, r in enumerate(runs)
    })
    pred_matrix['actual'] = y_true

    run_cols = [c for c in pred_matrix.columns if c.startswith('run_')]
    pred_matrix['all_agree'] = pred_matrix[run_cols].nunique(axis=1) == 1
    pred_matrix['majority_vote'] = pred_matrix[run_cols].mode(axis=1)[0].astype(int)
    pred_matrix['majority_correct'] = (pred_matrix['majority_vote'] == pred_matrix['actual']).astype(int)

    n_stable = pred_matrix['all_agree'].sum()
    n_unstable = len(pred_matrix) - n_stable

    print(f"  Stable (all {N_RUNS} runs agree): {n_stable}/100 ({n_stable}%)")
    print(f"  Unstable (at least one flip):     {n_unstable}/100 ({n_unstable}%)")
    print(f"  Majority vote accuracy: {pred_matrix['majority_correct'].mean()*100:.1f}%")

    # Per-run metrics
    run_metrics = []
    for i, r in enumerate(runs):
        m = r['metrics'].copy()
        m['run'] = i + 1
        run_metrics.append(m)

    metrics_df = pd.DataFrame(run_metrics).set_index('run')
    print(f"\n  Per-run metrics:")
    print(f"  {metrics_df[['accuracy','f1_charged_off']].to_string()}")
    print(f"  Accuracy range: {metrics_df['accuracy'].min()*100:.1f}% - {metrics_df['accuracy'].max()*100:.1f}%")
    print(f"  Accuracy std:   {metrics_df['accuracy'].std()*100:.2f}%")


Stability: GPT-5.4 (no_desc)
  Stable (all 3 runs agree): 95/100 (95%)
  Unstable (at least one flip):     5/100 (5%)
  Majority vote accuracy: 81.0%

  Per-run metrics:
       accuracy  f1_charged_off
run                          
1        0.78        0.266667
2        0.82        0.357143
3        0.82        0.357143
  Accuracy range: 78.0% - 82.0%
  Accuracy std:   2.31%

Stability: GPT-5.4 (with_desc)
  Stable (all 3 runs agree): 99/100 (99%)
  Unstable (at least one flip):     1/100 (1%)
  Majority vote accuracy: 77.0%

  Per-run metrics:
       accuracy  f1_charged_off
run                          
1        0.77        0.303030
2        0.77        0.303030
3        0.76        0.294118
  Accuracy range: 76.0% - 77.0%
  Accuracy std:   0.58%


In [6]:
# Per-run metrics comparison (both conditions)
for cond_name, runs in [('no_desc', runs_no_desc), ('with_desc', runs_with_desc)]:
    print(f"\n{'='*60}")
    print(f"Per-run metrics: {LABEL} ({cond_name})")
    print(f"{'='*60}")

    run_metrics = []
    for i, r in enumerate(runs):
        m = r['metrics'].copy()
        m['run'] = i + 1
        run_metrics.append(m)

    metrics_df = pd.DataFrame(run_metrics).set_index('run')
    print(metrics_df.to_string())

    print(f"\nAccuracy range: {metrics_df['accuracy'].min()*100:.1f}% - {metrics_df['accuracy'].max()*100:.1f}%")
    print(f"Accuracy std:   {metrics_df['accuracy'].std()*100:.2f}%")
    print(f"CO F1 range:    {metrics_df['f1_charged_off'].min():.3f} - {metrics_df['f1_charged_off'].max():.3f}")


Per-run metrics: GPT-5.4 (no_desc)
     accuracy  precision_charged_off  recall_charged_off  f1_charged_off       auc  n_valid
run                                                                                        
1        0.78               0.266667            0.266667        0.266667  0.594118      100
2        0.82               0.384615            0.333333        0.357143  0.631765      100
3        0.82               0.384615            0.333333        0.357143  0.639608      100

Accuracy range: 78.0% - 82.0%
Accuracy std:   2.31%
CO F1 range:    0.267 - 0.357

Per-run metrics: GPT-5.4 (with_desc)
     accuracy  precision_charged_off  recall_charged_off  f1_charged_off       auc  n_valid
run                                                                                        
1        0.77               0.277778            0.333333        0.303030  0.596078      100
2        0.77               0.277778            0.333333        0.303030  0.594118      100
3        0.76  

### The loans that flipped

List every unstable loan with its per-run votes and the majority call. These are mostly borderline cases — the link between confidence and stability is followed up in `01e`.

In [7]:
# Inspect unstable predictions (both conditions)
for cond_name, runs in [('no_desc', runs_no_desc), ('with_desc', runs_with_desc)]:
    print(f"\n{'='*60}")
    print(f"Unstable predictions: {LABEL} ({cond_name})")
    print(f"{'='*60}")

    pred_matrix = pd.DataFrame({
        f'run_{i+1}': r['predictions'] for i, r in enumerate(runs)
    })
    pred_matrix['actual'] = y_true

    run_cols = [c for c in pred_matrix.columns if c.startswith('run_')]
    pred_matrix['all_agree'] = pred_matrix[run_cols].nunique(axis=1) == 1
    pred_matrix['majority_vote'] = pred_matrix[run_cols].mode(axis=1)[0].astype(int)
    pred_matrix['majority_correct'] = (pred_matrix['majority_vote'] == pred_matrix['actual']).astype(int)

    unstable = pred_matrix[~pred_matrix['all_agree']].copy()
    if len(unstable) > 0:
        print(f"\n{len(unstable)} unstable samples:")
        for idx, row in unstable.iterrows():
            actual = 'Fully Paid' if row['actual'] == 1 else 'Charged Off'
            preds = [row[c] for c in run_cols]
            pred_str = ', '.join(['FP' if p == 1 else 'CO' for p in preds])
            majority = 'FP' if row['majority_vote'] == 1 else 'CO'
            correct = '✓' if row['majority_correct'] else '✗'
            print(f"  Sample {idx}: Actual={actual} | Runs=[{pred_str}] | Majority={majority} ({correct})")
    else:
        print("All predictions were perfectly stable across runs!")


Unstable predictions: GPT-5.4 (no_desc)

5 unstable samples:
  Sample 32: Actual=Charged Off | Runs=[FP, CO, CO] | Majority=CO (✓)
  Sample 51: Actual=Fully Paid | Runs=[CO, FP, CO] | Majority=CO (✗)
  Sample 52: Actual=Fully Paid | Runs=[CO, CO, FP] | Majority=CO (✗)
  Sample 55: Actual=Fully Paid | Runs=[CO, FP, FP] | Majority=FP (✓)
  Sample 71: Actual=Fully Paid | Runs=[CO, FP, FP] | Majority=FP (✓)

Unstable predictions: GPT-5.4 (with_desc)

1 unstable samples:
  Sample 55: Actual=Fully Paid | Runs=[FP, FP, CO] | Majority=FP (✓)


## Export Results

Write the per-loan run matrix + stability flags (`01b_predictions.csv`) and per-run metrics (`01b_metrics.csv`), with per-run cost embedded. Phase 2's `02a` reuses these no_desc runs as its 'Control' condition.

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

all_metrics = []
all_predictions = []

for cond_name, runs in [('no_desc', runs_no_desc), ('with_desc', runs_with_desc)]:
    pred_matrix = pd.DataFrame({
        f'run_{i+1}': r['predictions'] for i, r in enumerate(runs)
    })
    pred_matrix['actual'] = y_true

    run_cols = [c for c in pred_matrix.columns if c.startswith('run_')]
    pred_matrix['all_agree'] = pred_matrix[run_cols].nunique(axis=1) == 1
    pred_matrix['majority_vote'] = pred_matrix[run_cols].mode(axis=1)[0].astype(int)
    pred_matrix['majority_correct'] = (pred_matrix['majority_vote'] == pred_matrix['actual']).astype(int)

    # Embed per-run, per-loan cost so cost is derivable from this file alone.
    # Named cost_usd_run_* (NOT run_*) so it doesn't get picked up as a
    # prediction column by the run_cols logic above.
    for i, r in enumerate(runs):
        cu = r.get('cost_usd')
        if cu is not None:
            pred_matrix[f'cost_usd_run_{i+1}'] = cu

    pred_matrix.insert(0, 'desc_tag', cond_name)
    all_predictions.append(pred_matrix)

    for i, r in enumerate(runs):
        m = r['metrics'].copy()
        m['run'] = i + 1
        m['condition'] = cond_name
        all_metrics.append(m)

    n_stable = pred_matrix['all_agree'].sum()
    cond_metrics = [m for m in all_metrics if m['condition'] == cond_name]
    accs = [m['accuracy'] for m in cond_metrics]
    print(f"{cond_name}: {n_stable}% stable, "
          f"accuracy range {min(accs)*100:.1f}%-{max(accs)*100:.1f}%")

pd.concat(all_predictions, ignore_index=True).to_csv(
    f"{RESULTS_DIR}/01b_predictions.csv", index=False
)
pd.DataFrame(all_metrics).to_csv(f"{RESULTS_DIR}/01b_metrics.csv", index=False)

print(f"\nResults saved to {RESULTS_DIR}/")